In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
# import time
import imageio

In [2]:
def carregar_csv(caminho):
    df = pd.read_csv(caminho, header=None)

    # remove aspas + troca vírgula decimal por ponto
    df[0] = df[0].str.replace('"', '').str.replace(',', '.').astype(float)
    df[1] = df[1].str.replace('"', '').str.replace(',', '.').astype(float)
    df[2] = df[2].astype(int)

    dados = df[[0, 1]].values
    rotulos = df[2].values - 1

    return dados, rotulos

In [3]:
def gerar_dados():
    dados = []
    rotulos = []

    centros = [(-20, -20), (20, 20), (-20, 20), (20, -20)]

    for i, (cx, cy) in enumerate(centros):
        for _ in range(100):
            x = np.random.normal(cx, 2)
            y = np.random.normal(cy, 2)
            dados.append([x, y])
            rotulos.append(i)

    return np.array(dados), np.array(rotulos)

In [4]:
def normalizar(dados):
    min_vals = dados.min(axis=0)
    max_vals = dados.max(axis=0)
    return (dados - min_vals) / (max_vals - min_vals)

In [5]:
def distribuir_dados(grid, dados, rotulos):
    linhas, colunas = grid.shape
    indices = np.random.choice(linhas * colunas, len(dados), replace=False)
    xs, ys = np.unravel_index(indices, (linhas, colunas))

    for i in range(len(dados)):
        grid[xs[i], ys[i]] = (dados[i], rotulos[i])

In [6]:
def distancia(a, b):
    return np.linalg.norm(a - b)

In [7]:
class Ant:
    # def __init__(self, x, y, r=1, k1=0.1, k2=0.3):
    def __init__(self, x, y, r=1, k1=0.3, k2=0.6, alpha=0.1):

        self.x = x
        self.y = y
        self.r = r

        self.carregando = False 

        self.k1 = k1
        self.k2 = k2

        self.alpha = alpha

        self.item = None
        

    def move(self, max_x, max_y):
        direcoes = [(0, 1), (0, -1), (1, 0), (-1, 0)]
        dx, dy = random.choice(direcoes)
        
        self.x = (self.x + dx) % max_x
        self.y = (self.y + dy) % max_y
            
        # self.x, self.y 


    # def get_density(self, matrix):
    #     linhas, colunas = matrix.shape
    #     n = 0
    #     total_celulas = 0
        
    #     for i in range(self.x - self.r, self.x + self.r + 1):
    #         for j in range(self.y - self.r, self.y + self.r + 1):
                
    #             ii = i % linhas
    #             jj = j % colunas

    #             n += matrix[ii, jj]
    #             total_celulas += 1
    
    #     return n / total_celulas if total_celulas > 0 else 0
    def get_density(self, matrix):
        linhas, colunas = matrix.shape
        vizinhos = []

        for i in range(self.x - self.r, self.x + self.r + 1):
            for j in range(self.y - self.r, self.y + self.r + 1):
                ii = i % linhas
                jj = j % colunas

                if matrix[ii, jj] is not None:
                    vizinhos.append(matrix[ii, jj][0])

        # if self.carregando:
        #     item = self.item
        if self.carregando:
            item = self.item[0]
        else:
            if matrix[self.x, self.y] is None:
                return 0
            item = matrix[self.x, self.y][0]

        if len(vizinhos) == 0:
            return 0

        soma = 0
        for v in vizinhos:
            d = distancia(item, v)
            soma += max(0, 1 - (d / self.alpha))

        return soma / len(vizinhos)

    def decidir_acao(self, matrix):
        item_na_celula = matrix[self.x, self.y]

        # PEGAR
        if not self.carregando:
            if item_na_celula is not None:
                f = self.get_density(matrix)
                p_pegar = (self.k1 / (self.k1 + f))**2

                if random.random() < p_pegar:
                    self.item = item_na_celula
                    matrix[self.x, self.y] = None
                    self.carregando = True

        # LARGAR
        else:
            if item_na_celula is None:
                f = self.get_density(matrix)
                p_largar = (f / (self.k2 + f))**2

                if random.random() < p_largar:
                    matrix[self.x, self.y] = self.item
                    self.item = None
                    self.carregando = False

In [8]:
def visualizar(grid):
    plt.figure(figsize=(6,6))
    cores = ['red', 'blue', 'green', 'purple']

    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if grid[i, j] is not None:
                _, label = grid[i, j]
                plt.scatter(i, j, color=cores[label], s=10)

    plt.title("Clusters formados")
    plt.show()

In [9]:
def renderizar_grid(grid, scale=6):
    img = np.zeros((grid.shape[0], grid.shape[1], 3), dtype=np.uint8)

    cores = [
        [255, 0, 0],    # vermelho
        [0, 0, 255],    # azul
        [0, 255, 0],    # verde
        [128, 0, 128]   # roxo
    ]

    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if grid[i, j] is not None:
                _, label = grid[i, j]
                img[i, j] = cores[label]

    # aumenta resolução
    img = np.kron(img, np.ones((scale, scale, 1)))

    return img.astype(np.uint8)

In [10]:
# dados, rotulos = carregar_csv("Base.csv")

# print(dados.shape)
# print(rotulos.shape)

In [12]:
LARGURA = 50
ALTURA = 50
NUM_FORMIGAS = 40
MAX_INTERACOES = 900000

# gerar dados
# dados, rotulos = gerar_dados()
# dados, rotulos = carregar_csv("Base.csv")
dados, rotulos = carregar_csv("Base.csv")
# print(len(dados))
dados = normalizar(dados)

# criar grid
grid = np.empty((LARGURA, ALTURA), dtype=object)
grid[:] = None

distribuir_dados(grid, dados, rotulos)

# criar formigas
formigas = [
    Ant(random.randint(0, LARGURA-1),
        random.randint(0, ALTURA-1),
        r=2, k1=0.2, k2=0.3, alpha=0.1)
    for _ in range(NUM_FORMIGAS)
]

print("Simulação iniciando...")

frame_inicial = renderizar_grid(grid)
imageio.imwrite("estado_inicial.png", frame_inicial)

with imageio.get_writer("ant_clustering.gif", mode="I", fps=5) as writer:

    for it in range(MAX_INTERACOES):
        random.shuffle(formigas)

        for ant in formigas:
            ant.move(LARGURA, ALTURA)
            ant.decidir_acao(grid)

        if it % 2000 == 0:
            print(f"Frame {it}")
            frame = renderizar_grid(grid)
            writer.append_data(frame)

frame_final = renderizar_grid(grid)
imageio.imwrite("estado_final.png", frame_final)

print("Simulação finalizada!")


Simulação iniciando...
Frame 0
Frame 2000
Frame 4000
Frame 6000
Frame 8000
Frame 10000
Frame 12000
Frame 14000
Frame 16000
Frame 18000
Frame 20000
Frame 22000
Frame 24000
Frame 26000
Frame 28000
Frame 30000
Frame 32000
Frame 34000
Frame 36000
Frame 38000
Frame 40000
Frame 42000
Frame 44000
Frame 46000
Frame 48000
Frame 50000
Frame 52000
Frame 54000
Frame 56000
Frame 58000
Frame 60000
Frame 62000
Frame 64000
Frame 66000
Frame 68000
Frame 70000
Frame 72000
Frame 74000
Frame 76000
Frame 78000
Frame 80000
Frame 82000
Frame 84000
Frame 86000
Frame 88000
Frame 90000
Frame 92000
Frame 94000
Frame 96000
Frame 98000
Frame 100000
Frame 102000
Frame 104000
Frame 106000
Frame 108000
Frame 110000
Frame 112000
Frame 114000
Frame 116000
Frame 118000
Frame 120000
Frame 122000
Frame 124000
Frame 126000
Frame 128000
Frame 130000
Frame 132000
Frame 134000
Frame 136000
Frame 138000
Frame 140000
Frame 142000
Frame 144000
Frame 146000
Frame 148000
Frame 150000
Frame 152000
Frame 154000
Frame 156000
Frame 15